In [ ]:
import os
import json
from collections import defaultdict

directory = "results"  

aggregated = defaultdict(lambda: defaultdict(lambda: {"win": 0, "lose": 0, "tie": 0}))

for filename in os.listdir(directory):
    if filename.endswith(".json"):
        filepath = os.path.join(directory, filename)
        with open(filepath, "r") as f:
            data = json.load(f)
        
        for model_name, model_results in data.items():
            for metric, scores in model_results.items():
                aggregated[model_name][metric]["win"]  += scores["win"]
                aggregated[model_name][metric]["lose"] += scores["lose"]
                aggregated[model_name][metric]["tie"]  += scores["tie"]

for model_name, metrics in aggregated.items():
    print(f"\nModel: {model_name}")
    for metric, result in metrics.items():
        total = result["win"] + result["lose"] + result["tie"]
        print(f"  {metric}: win={result['win']} ({result['win']/total*100:.2f}%), lose={result['lose']} ({result['lose']/total*100:.2f}%), tie={result['tie']} ({result['tie']/total*100:.2f}%)")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib import font_manager as fm
from collections import OrderedDict

font_path = "./Helvetica.ttf"  
helvetica_font = fm.FontProperties(fname=font_path)
font_name = helvetica_font.get_name()

bold_font_path = "./Helvetica-Bold.ttf"  
helvetica_bold = fm.FontProperties(fname=bold_font_path)

font_size = 16
mpl.rcParams.update({
    'font.size': font_size,       
    'axes.titlesize': font_size,  
    'axes.labelsize': font_size,  
    'xtick.labelsize': font_size,
    'ytick.labelsize': font_size, 
    'legend.fontsize': font_size 
})

def plot_horizontal_win_tie_lose_with_percentages(methods, questions, win_counts, tie_counts, lose_counts):
    fig, axes = plt.subplots(len(questions), 1, figsize=(10, 6), sharex=True)

    if len(questions) == 1:
        axes = [axes]

    win_color  = "#1E88E5"  # Blue
    tie_color  = "#F9A825"  # Amber
    lose_color = "#D32F2F"  # Deep Red

    for i, q in enumerate(questions):
        ax = axes[i]
        y_pos = np.arange(len(methods))

        total = win_counts[:, i] + tie_counts[:, i] + lose_counts[:, i]
        win_pct = win_counts[:, i] / total * 100
        tie_pct = tie_counts[:, i] / total * 100
        lose_pct = lose_counts[:, i] / total * 100

        left = np.zeros(len(methods))
        bars_info = []  # 각 bar의 left와 width를 저장
        for counts, pct, label, color in zip(
            [lose_counts[:, i], tie_counts[:, i], win_counts[:, i]],  # 순서 변경
            [lose_pct, tie_pct, win_pct],                             # 퍼센트도 동일하게
            ["Lose", "Tie", "Win"],                                   # 레이블 순서 맞춤
            [lose_color, tie_color, win_color]                        # 색상 순서 맞춤
        ):
            bars = ax.barh(y_pos, counts, left=left, label=label if i == 0 else "", color=color)
            for bar, p in zip(bars, pct):
                if p > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_y() + bar.get_height() / 2,
                            f"{p:.0f}%", ha='center', va='center',
                            color='white', fontsize=font_size, fontweight='bold',
                            fontproperties=helvetica_bold)
            bars_info.append((left.copy(), counts.copy()))  # 경계 계산용
            left += counts
          
        # Lose와 Tie 사이 경계선 (각 row마다)
        lose_left_array, lose_width_array = bars_info[0]
        for y, (lose_left, lose_width) in enumerate(zip(lose_left_array, lose_width_array)):
            boundary_x = lose_left + lose_width
            ax.plot([boundary_x, boundary_x],
                    [y - 0.4, y + 0.4],
                    color='black', linewidth=3, label=None)  # label=None으로 legend 방지

        # y축 설정 및 제목
        ax.set_yticks(y_pos)
        ax.set_yticklabels(methods, fontproperties=helvetica_bold, fontsize=font_size)
        ax.tick_params(axis='y', length=0)
        ax.set_title(f"{q}", fontproperties=helvetica_bold, fontsize=font_size, fontweight='bold')

        # x축과 스파인 숨김
        ax.xaxis.set_visible(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        

    # 범례를 색상 막대만 다시 지정 (Lose/Tie/Win만)
    handles = [
        plt.Line2D([0], [0], color=lose_color, lw=10),
        plt.Line2D([0], [0], color=tie_color, lw=10),
        plt.Line2D([0], [0], color=win_color, lw=10)
    ]
    fig.legend(handles, ["Win", "Tie", "Lose"], loc='upper center', ncol=3,
            bbox_to_anchor=(0.5, 0.05), prop=helvetica_bold, fontsize=font_size,
            handlelength=2.0, handleheight=1,
           borderpad=0.6)      

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    
    plt.savefig("human.pdf", bbox_inches='tight')            
    
    plt.show()




# methods와 questions 추출
methods = list(aggregated.keys())

desired_questions = ["context_groundedness", "new_image_description"]
desired_order = ['chatgpt', 'gemini', 'base']

from collections import OrderedDict
for model in aggregated:
    metrics = aggregated[model]
    # 원하는 순서로 재정렬
    new_order = OrderedDict()
    for key in desired_questions:
        new_order[key] = metrics[key]
    aggregated[model] = new_order

# OrderedDict로 재정렬
aggregated_ordered = OrderedDict((k, aggregated[k]) for k in desired_order)

# 배열 초기화 (methods × questions)
win_counts = np.zeros((len(methods), len(desired_questions)), dtype=int)
tie_counts = np.zeros((len(methods), len(desired_questions)), dtype=int)
lose_counts = np.zeros((len(methods), len(desired_questions)), dtype=int)

for m_idx, (model_name, metrics) in enumerate(aggregated_ordered.items()):
    for q_idx, q in enumerate(desired_questions):
        print(q)
        result = metrics[q]
        total = result["win"] + result["tie"] + result["lose"]
        
        win_pct = round(result["lose"] / total * 100)
        lose_pct = round(result["win"] / total * 100)
        tie_pct = 100 - (win_pct + lose_pct)
        
        win_counts[m_idx, q_idx] = win_pct
        tie_counts[m_idx, q_idx] = tie_pct
        lose_counts[m_idx, q_idx] = lose_pct

methods = ["GPT-5", "Gemini-3.0 Pro", "Qwen3-VL 8B"]
questions = ["Context Groundedness", "New Image Description"]


plot_horizontal_win_tie_lose_with_percentages(methods, questions, win_counts, tie_counts, lose_counts)
